<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/llama_cpp_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# llama.cpp on a Colab T4

Build [llama.cpp](https://github.com/ggml-org/llama.cpp) with CUDA, download a GGUF model, and chat with it.

Runtime → Change runtime type → **T4 GPU**. The first build takes several minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Config

In [ ]:
# Qwen3.8-27B, Unsloth dynamic Q4_K_M (~17.1 GB). Single-file quant.
REPO_ID   = "unsloth/Qwen3.8-27B-GGUF"
FILENAME  = "Qwen3.8-27B-UD-Q4_K_M.gguf"
# Smaller alternatives if this is too slow or runs out of memory:
#   FILENAME = "Qwen3.8-27B-UD-IQ4_XS.gguf"      (~15.7 GB, almost fits the T4 entirely)
#   REPO_ID = "bartowski/Qwen3.8-27B-GGUF"; FILENAME = "Qwen3.8-27B-Q4_K_M.gguf"

# The Q4_K_M file (17.1 GB) is larger than the T4's 16 GB, so only part of the model goes on the GPU.
# The remaining layers run on the CPU out of system RAM (~12 GB), which is much slower.
# Raise N_GPU_LAYERS until you hit an out-of-memory error, then back off a few layers.
N_GPU_LAYERS = 48     # the model has about 64 layers; 99 would offload all and fail on a T4
CTX_SIZE     = 4096   # more context = more VRAM
REBUILD_LLAMA = False   # True = ignore the cached build, pull the latest llama.cpp, rebuild, and re-save it

## 3. Get llama.cpp: restore the build from Google Drive (first run builds with CUDA for the T4, compute capability 7.5, and saves it)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

BUILD_DIR = "/content/drive/MyDrive/llm_models/builds"
os.makedirs(BUILD_DIR, exist_ok=True)
TARBALL = f"{BUILD_DIR}/llama.cpp_cuda_sm75.tar"
SRC, BIN = "/content/llama.cpp", "/content/llama.cpp/build/bin"

def works():
    try:
        return subprocess.run([f"{BIN}/llama-server", "--version"], capture_output=True, timeout=120).returncode == 0
    except Exception:
        return False

if not REBUILD_LLAMA and os.path.exists(TARBALL):
    print("Restoring cached build from Drive...")
    os.makedirs(f"{SRC}/build", exist_ok=True)
    subprocess.run(["tar", "-xf", TARBALL, "-C", f"{SRC}/build"], check=True)

if works():
    print("Cached build works.")
else:
    print("No usable cached build; building (several minutes)...")
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        open("/content/llama_build.log", "a").write(r.stdout + r.stderr)
        if r.returncode:
            raise RuntimeError(f"Failed: {cmd}\n" + (r.stdout + r.stderr)[-3000:])
    open("/content/llama_build.log", "w").close()
    sh(f"rm -rf {SRC} && git clone --depth 1 https://github.com/ggml-org/llama.cpp {SRC}")
    sh(f"cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF")
    sh(f"cd {SRC} && cmake --build build --config Release -j $(nproc) --target llama-server llama-cli")
    rev = subprocess.run(f"cd {SRC} && git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
    open(f"{SRC}/build/bin/BUILD_INFO.txt", "w").write(f"llama.cpp {rev}\n")
    subprocess.run(["tar", "-cf", TARBALL + ".part", "-C", f"{SRC}/build", "bin"], check=True)
    os.rename(TARBALL + ".part", TARBALL)
    print("Saved build to Drive:", TARBALL)

In [ ]:
print(os.listdir(BIN)[:20])
assert os.path.exists(f"{BIN}/llama-server"), "Build failed; see /content/llama_build.log"
print(open(f"{BIN}/BUILD_INFO.txt").read() if os.path.exists(f"{BIN}/BUILD_INFO.txt") else "")

## 4. Get the model (from Google Drive if saved, otherwise download once and save)

In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/llm_models"   # permanent copy
LOCAL_DIR = "/content/models"                     # fast local disk; llama.cpp loads from here
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(LOCAL_DIR, exist_ok=True)
drive_path = f"{DRIVE_DIR}/{FILENAME}"
MODEL_PATH = f"{LOCAL_DIR}/{FILENAME}"

if not os.path.exists(drive_path):
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    print("Not on Drive yet; downloading from Hugging Face...")
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=LOCAL_DIR)
    shutil.copyfile(MODEL_PATH, drive_path + ".part")   # .part so a cut-off copy is never mistaken for a full file
    os.rename(drive_path + ".part", drive_path)
    print("Saved to Drive:", drive_path)
elif not os.path.exists(MODEL_PATH):
    print("Copying from Drive to local disk (faster to load than reading from Drive)...")
    shutil.copyfile(drive_path, MODEL_PATH)
print(MODEL_PATH, f"{os.path.getsize(MODEL_PATH)/1e9:.1f} GB")
!df -h /content | tail -1

## 5. Start the server in the background (also serves a built-in chat web UI)

In [ ]:
import subprocess, time, requests, secrets

API_KEY = secrets.token_urlsafe(24)   # required by every request, including through the public tunnel

server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server",
     "-m", MODEL_PATH, "-ngl", str(N_GPU_LAYERS), "-c", str(CTX_SIZE),
     "--host", "127.0.0.1", "--port", "8080", "--api-key", API_KEY],
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"}, json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    m = r.json()["choices"][0]["message"]
    return m.get("reasoning_content") and ("[thinking]\n" + m["reasoning_content"] + "\n[answer]\n" + m["content"]) or m["content"]

print(chat("Explain what a GGUF file is in two sentences."))

## 7. Open the web UI and the API from outside Colab

llama-server has its own chat web UI at `/`, the same idea as Unsloth Studio. This opens a free Cloudflare tunnel so you can use it from your browser, phone, or any OpenAI-compatible client.
The URL is public but every request needs the API key. The UI asks for it under Settings. Anyone with both the URL and key can use your GPU, so don't share them.

In [ ]:
import subprocess, re, time, os, urllib.request

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

def open_tunnel(port):
    """Expose a local port through a free Cloudflare quick tunnel and return (process, public URL)."""
    log_path = f"/content/cloudflared_{port}.log"
    proc = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                            stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError("No tunnel URL yet:\n" + open(log_path).read()[-2000:])

In [ ]:
tunnel, PUBLIC_URL = open_tunnel(8080)
print("Web UI :", PUBLIC_URL)
print("API    :", PUBLIC_URL + "/v1")
print("API key:", API_KEY)
print(f"""
# From any machine:
from openai import OpenAI
client = OpenAI(base_url="{PUBLIC_URL}/v1", api_key="{API_KEY}")
print(client.chat.completions.create(model="local", messages=[{{"role": "user", "content": "Hello"}}]).choices[0].message.content)

# Or with curl:
curl {PUBLIC_URL}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
  -d '{{"messages":[{{"role":"user","content":"Hello"}}]}}'
""")

## 8. Cleanup and VRAM check

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
# server.terminate()   # stop the server when finished

## Notes
- Qwen3.8-27B is a hybrid thinking model, so replies may start with a reasoning block and use many tokens.
- The 17.1 GB Q4_K_M file does not fit in 16 GB of VRAM, so some layers run on the CPU. Expect a few tokens per second.
- The model is saved to `MyDrive/llm_models` on Google Drive, so later sessions copy it from Drive instead of downloading it again.
- The tunnel and API key last only for this session. Run the tunnel cell again for a new URL after a restart.
- The compiled llama.cpp is cached on Drive (`MyDrive/llm_models/builds`). Set `REBUILD_LLAMA = True` to pick up a newer llama.cpp.
- If the build fails, read `/content/llama_build.log`.